# AML - SAIGE-GENE SET-BASED - ASSOCIATION ANALYSIS
This script performs rare variant set-based gene association tests for each chromosome using SAIGE-GENE+. It is configured to use the sparse GRM with 2,000 randomly selected markers to estimate the variance ratio. Maximum minor allele frequency thresholds for each test are set at 1%, 0.1%, and 0.001%. The script evaluates the annotation groups Missense, Loss-of-Function, and a combination of Missense and Loss-of-Function. An additional analysis is conducted using only synonymous variants; this is executed with a separate command so that the Cauchy values from the first test reflect only predicted deleterious variants (Missense and Loss-of-Function).

In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


# COMMAND 

In [ ]:
%%writefile high_confidence_deprivation_index_analysis.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in aml_phenotype; do 
    echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}"
    
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_high_confidence_chr${chrom}_deprivation_index" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_deprivation_index_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_deprivation_index_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --groupFile="${vat}/${pop}_HC_annotation/HC-annotation-chr${chrom}.txt" \
        --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
        --maxMAF_in_groupTest=0.0001,0.001,0.01 \
        --is_output_markerList_in_groupTest=TRUE \
        --is_fastTest=TRUE
                
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_synonymous_chr${chrom}_deprivation_index" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_deprivation_index_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_deprivation_index_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --groupFile="${vat}/${pop}_HC_annotation/HC-annotation-chr${chrom}.txt" \
        --annotation_in_groupTest=Synonymous \
        --maxMAF_in_groupTest=0.0001,0.001,0.01 \
        --is_output_markerList_in_groupTest=TRUE \
        --is_fastTest=TRUE
                
    done
done


In [ ]:
!gsutil cp high_confidence_deprivation_index_analysis.sh {my_bucket}/data/scripts/

In [ ]:
!gsutil cat {my_bucket}/data/scripts/high_confidence_deprivation_index_analysis.sh

# SUBMIT COMMAND 

In [ ]:
%%bash --out saige_gene_step02

LOWER=1
UPPER=2
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 440 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=2
UPPER=3
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 300 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=3
UPPER=4
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 280 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=4
UPPER=5
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 250 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=5
UPPER=7
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 230 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=7
UPPER=8
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 230 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=8
UPPER=9
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 180 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=9
UPPER=11
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 190 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=11
UPPER=13
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 260 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=13
UPPER=14
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 110 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=14
UPPER=15
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 160 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=15
UPPER=16
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 175 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=16
UPPER=17
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 210 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=17
UPPER=18
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 250 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=18
UPPER=19
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 110 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=19
UPPER=20
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 275 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=20
UPPER=21
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 130 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=21
UPPER=22
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 60 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

In [ ]:
%%bash --out saige_gene_step02

LOWER=22
UPPER=23
for ((chromo=$LOWER;chromo<$UPPER;chromo+=1))
do
source ~/aou_dsub.bash # This file was created via notebook 01_dsub_setup.ipynb.

mnt_path="/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed"

aou_dsub \
  --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
  --boot-disk-size 10 \
  --disk-size 110 \
  --min-ram 4 \
  --logging "${WORKSPACE_BUCKET}/data/logging" \
  --input plink_bed="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bed" \
  --input plink_bim="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.bim" \
  --input plink_fam="gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr${chromo}.fam" \
  --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
  --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
  --input-recursive vat="${WORKSPACE_BUCKET}/data/annotation_table/" \
  --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/" \
  --env genos="${mnt_path}" \
  --env chrom=${chromo} \
  --script "${WORKSPACE_BUCKET}/data/scripts/high_confidence_deprivation_index_analysis.sh"

done

## DOWNLOAD RESULTS FROM THE WORKSPACE BUCKET 

In [ ]:
!rm -rf saige_gene_step02
!rm -rf merged_saige_gene_step02

In [ ]:
!mkdir -p saige_gene_step02
!mkdir -p merged_saige_gene_step02

In [ ]:
!gsutil cp {my_bucket}/data/saige_step02/HC_analysis/*_deprivation_index saige_gene_step02

In [ ]:
%%bash 

# ========================================================================================
# Format gene results for download  
# ========================================================================================

mkdir -p redacted_saige_gene_step02
unmerge_files=saige_gene_step02
redacted_files=redacted_saige_gene_step02

for pheno in aml_phenotype_high_confidence aml_phenotype_synonymous; do 
    for chrom in {1..22}; do
        echo "Processing phenotype: ${pheno} | chr: ${chrom}"
        # Redact each chromosome file 
        awk -v chrom=$chrom 'BEGIN {OFS="\t"} 
            NR==1 {print "Chromosome", $1, $2, $3, $4, $5, $6, $7; next} 
            {print chrom, $1, $2, $3, $4, $5, $6, $7}' \
            $unmerge_files/EUR_${pheno}_chr${chrom}_deprivation_index  > \
            $redacted_files/EUR_${pheno}_redacted_chr${chrom}_deprivation_index 
    done
done


In [ ]:
%%bash 

# ========================================================================================
# Format gene results for plotting 
# ========================================================================================

mkdir -p merged_saige_gene_step02
unmerge_files=saige_gene_step02
merge_files=merged_saige_gene_step02
for pheno in aml_phenotype_high_confidence aml_phenotype_synonymous; do 
    echo "Processing phenotype: ${pheno} | chr: 1"
    awk 'BEGIN {OFS="\t"} 
         NR==1 {print $0, "Chromosome"; next} 
        {print $0, 1}' $unmerge_files/EUR_${pheno}_chr1_deprivation_index  > $merge_files/EUR_${pheno}_merged_chr_deprivation_index 
    for chrom in {2..22}; do  # Start from chr2 since chr1 is already used above
         echo "Processing phenotype: ${pheno} | chr: ${chrom}"
         awk -v chrom=$chrom 'BEGIN {OFS="\t"} 
                              NR==1 {print $0, "Chromosome"; next} 
                              {print $0, chrom}' $unmerge_files/EUR_${pheno}_chr${chrom}_deprivation_index  > $unmerge_files/EUR_${pheno}_formatted_chr${chrom}_deprivation_index 
        tail +2 $unmerge_files/EUR_${pheno}_formatted_chr${chrom}_deprivation_index  >> $merge_files/EUR_${pheno}_merged_chr_deprivation_index 
    done
done


In [ ]:
import glob
import os
import pandas as pd 
import mygene
import re

# ========================================================================================
# Merge together results across chromosomes 
# ========================================================================================
files = []
for chrom in range(1,23):
    file = pd.read_csv(f'saige_gene_step02/EUR_aml_phenotype_high_confidence_chr{chrom}_deprivation_index',
                      sep = '\t')
    files.append(file)
    
files = pd.concat(files, axis = 0)

# ========================================================================================
# Format table of significant gene results 
# ========================================================================================
non_cauchy_pvalues = files[~(files['Group'] == 'Cauchy')]
non_cauchy_pvalues = non_cauchy_pvalues[['Region', 'Group', 'max_MAF', 
                                         'Pvalue', 'Pvalue_Burden', 'Pvalue_SKAT', 
                                         'BETA_Burden', 'SE_Burden']]

non_cauchy_pvalues['Bonferroni SKAT-O P-value'] = (
    (non_cauchy_pvalues['Pvalue'] * len(non_cauchy_pvalues))
)

non_cauchy_pvalues = (
    non_cauchy_pvalues[non_cauchy_pvalues['Pvalue'] < (0.05/len(non_cauchy_pvalues))]
                       )

mg = mygene.MyGeneInfo()

result = mg.querymany(non_cauchy_pvalues['Region'], 
                      scopes='ensembl.gene', 
                      fields='symbol', 
                      species='human')
symbols = []
for r in result: 
     symbols.append(r.get('symbol'))
symbols = pd.DataFrame(symbols, columns=["Gene Symbol"])

non_cauchy_pvalues['Gene Symbol'] = symbols['Gene Symbol'].values

non_cauchy_pvalues['Group'] = (
    non_cauchy_pvalues['Group'].str.replace(r';', ' & ', regex=True)
)

non_cauchy_pvalues = non_cauchy_pvalues.rename(columns={
    'max_MAF': 'Maximum MAF',
    'Pvalue': 'SKAT-O P-value',
    'Pvalue_Burden': 'Burden P-value',
    'Pvalue_SKAT': 'SKAT P-value',
    'BETA_Burden': 'Burden Beta',
    'SE_Burden': 'Burden Standard Error',
    'Group': 'Annotation Group'
})

non_cauchy_pvalues = (
    non_cauchy_pvalues[['Gene Symbol', 'Annotation Group', 'Maximum MAF', 
       'SKAT-O P-value', 'Bonferroni SKAT-O P-value','Burden P-value', 'SKAT P-value', 
       'Burden Beta','Burden Standard Error']]
)
    
# ========================================================================================
# Format table of significant cauchy gene results 
# ========================================================================================    
cauchy_pvalues = files[(files['Group'] == 'Cauchy')]

cauchy_pvalues['Bonferroni Cauchy SKAT-O P-value'] = (
    (cauchy_pvalues['Pvalue'] * len(cauchy_pvalues))
)

cauchy_pvalues = (
    cauchy_pvalues[cauchy_pvalues['Pvalue'] < (0.05/len(cauchy_pvalues))]
                       )

mg = mygene.MyGeneInfo()

result = mg.querymany(cauchy_pvalues['Region'], 
                      scopes='ensembl.gene', 
                      fields='symbol', 
                      species='human')
symbols = []
for r in result: 
     symbols.append(r.get('symbol'))
symbols = pd.DataFrame(symbols, columns=["Gene Symbol"])

cauchy_pvalues['Gene Symbol'] = symbols['Gene Symbol'].values


cauchy_pvalues = cauchy_pvalues.rename(columns={
    'max_MAF': 'Maximum MAF',
    'Pvalue': 'Cauchy SKAT-O P-value',
    'Pvalue_Burden': 'Cauchy Burden P-value',
    'Pvalue_SKAT': 'Cauchy SKAT P-value',
    'BETA_Burden': 'Burden Beta',
    'SE_Burden': 'Burden Standard Error',
    'Group': 'Annotation Group'
})

cauchy_pvalues = (
    cauchy_pvalues[['Gene Symbol',
       'Cauchy SKAT-O P-value', 
       'Bonferroni Cauchy SKAT-O P-value',
       'Cauchy Burden P-value', 
       'Cauchy SKAT P-value']]
)

# ========================================================================================
# Save significant gene results to excel files 
# ========================================================================================
non_cauchy_pvalues.to_excel("eur_aml_phenotype_deprivation_index_non_cauchy_pvalues.xlsx", index=False)
non_cauchy_pvalues.to_csv("eur_aml_phenotype_deprivation_index_non_cauchy_pvalues.tsv", 
                          sep ='\t', index=False)
cauchy_pvalues.to_excel("eur_aml_phenotype_deprivation_index_cauchy_pvalues.xlsx", index=False)
cauchy_pvalues.to_csv("eur_aml_phenotype_deprivation_index_cauchy_pvalues.tsv", 
                        sep ='\t', index=False)